# Model architecture

Walks through the 3D U-Net defined in `src/model.py` (MONAI `UNet`):

1. What a U-Net is and how this one is configured
2. What `channels` and `strides` mean, on a small 2D example
3. Every convolution in the network: shapes, channels, parameters
4. Memory: parameters and activations
5. Output: logits, sigmoid and why not softmax

In [2]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.model import IN_CHANNELS, build_unet, count_parameters

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PATCH = 128 if DEVICE.type == "cuda" else 64
print("device:", DEVICE, "| patch:", PATCH)

device: cpu | patch: 64


/home/student2/miniconda3/envs/brats/lib/python3.13/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 1. U-Net architecture used in this project

```
input 4 × 128³
  │ encoder (left side): resolution ↓, channels ↑
  ├─ level 1: 128³,  32 ch ─────────── skip ──────────┐
  ├─ level 2:  64³,  64 ch ──────── skip ──────┐      │
  ├─ level 3:  32³, 128 ch ───── skip ───┐     │      │
  ├─ level 4:  16³, 256 ch ── skip ──┐   │     │      │
  └─ level 5:   8³, 320 ch (bottom)  │   │     │      │
                                     │   │     │      │
  decoder (right side): resolution ↑, channels ↓      │
     16³ ◄── concat ◄────────────────┘   │     │      │
     32³ ◄── concat ◄────────────────────┘     │      │
     64³ ◄── concat ◄──────────────────────────┘      │
    128³ ◄── concat ◄─────────────────────────────────┘
output 4 × 128³ (ET, TC, WT, RC logits)
```

- The **encoder** halves the resolution at each level, so deeper layers combine information from a larger area.
- The **decoder** brings the resolution back up.
- **Skip connections** pass fine-detail feature maps from the encoder straight to the decoder at the same resolution, so precise boundaries are not lost at the bottom of the "U".

Configuration used here:

In [3]:
model = build_unet().to(DEVICE).eval()

cfg = {
    "in_channels": IN_CHANNELS,
    "out_channels": model.out_channels,
    "channels": model.channels,
    "strides": model.strides,
    "num_res_units": model.num_res_units,
    "parameters": f"{count_parameters(model) / 1e6:.2f} M",
}

pd.Series(cfg, name="value").to_frame()

,value
in_channels,4
out_channels,4
channels,"(32, 64, 128, 256, 320)"
strides,"(2, 2, 2, 2)"
num_res_units,2
parameters,12.87 M


## 2. `channels` and `strides` on a small 2D example

- **channels**: number of filters in a layer. Each filter is a separate learned convolution
  that spans *all* input channels and produces one output map. 32 filters $\rightarrow$ 32 output channels.
- **stride**: step of the filter. Stride 1 places a filter centre on every voxel; stride 2 on every
  second voxel in each dimension, i.e. on a sub-lattice, so the output is half the size.
- **padding**: the input is surrounded with zeros (one voxel for a 3×3 filter), so filters centred
  on the border still fit. With stride 1 this keeps the size unchanged.

Output size: `out = floor((n + 2·padding − kernel) / stride) + 1`.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# colours used in the plot and in the legend
PADDING = "#cccccc"
IMAGE = "#e6e6e6"
CENTRE = "#a82424"
FILTER = "#c56d6d"
OUTPUT = "#f8eded"

n, k, pad = 8, 3, 1
size = n + 2 * pad
grid = np.zeros((size, size))
grid[pad:pad + n, pad:pad + n] = 1  # 0 = zero padding, 1 = image
strides = (1, 2)
outs = [(n + 2 * pad - k) // s + 1 for s in strides]

fig = make_subplots(
    rows=2, cols=2, horizontal_spacing=0.06, vertical_spacing=0.08,
    subplot_titles=[t for s, o in zip(strides, outs)
                    for t in (f"<b>stride {s}: input {n}×{n} + padding</b>", f"<b>output {o}×{o}</b>")],
)

for row, (stride, out) in enumerate(zip(strides, outs), start=1):
    # input with padding
    fig.add_trace(go.Heatmap(z=grid, xgap=3, ygap=3, showscale=False, hoverinfo="skip",
                             colorscale=[[0, PADDING], [0.5, PADDING], [0.5, IMAGE], [1, IMAGE]]),
                  row=row, col=1)
    c = np.arange(out) * stride + pad
    cx, cy = np.meshgrid(c, c)
    fig.add_trace(go.Scatter(x=cx.ravel(), y=cy.ravel(), mode="markers", showlegend=False,
                             marker=dict(size=9, color=CENTRE), hoverinfo="skip"), row=row, col=1)
    
    centre = pad + stride  # filter centred on the second centre in each direction
    fig.add_shape(type="rect", x0=centre - k // 2 - 0.5, x1=centre + k // 2 + 0.5,
                  y0=centre - k // 2 - 0.5, y1=centre + k // 2 + 0.5,
                  line=dict(color=FILTER, width=4), row=row, col=1)

    # output at the same cell size, centred
    fig.add_trace(go.Heatmap(z=np.ones((out, out)), xgap=3, ygap=3, showscale=False, hoverinfo="skip",
                             colorscale=[[0, OUTPUT], [1, OUTPUT]]), row=row, col=2)
    fig.add_shape(type="rect", x0=0.5, x1=1.5, y0=0.5, y1=1.5, fillcolor=FILTER, opacity=0.85,
                  line=dict(width=0), row=row, col=2)  # output (1, 1)
    off = (size - out) / 2
    fig.update_xaxes(range=[-0.5 - off, out - 0.5 + off], row=row, col=2)
    fig.update_yaxes(range=[out - 0.5 + off, -0.5 - off], row=row, col=2)
    fig.update_yaxes(autorange="reversed", row=row, col=1)

# equal aspect in every panel, no axes
for i in range(1, 5):
    sx = "" if i == 1 else str(i)
    fig.layout[f"yaxis{sx}"].update(scaleanchor=f"x{sx}", scaleratio=1)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)

# legend from dummy traces
legend = [
    ("zero padding", dict(symbol="square", size=16, color=PADDING)),
    ("input image", dict(symbol="square", size=16, color=IMAGE)),
    ("filter centres", dict(symbol="circle", size=10, color=CENTRE)),
    ("one 3×3 filter", dict(symbol="square-open", size=16, color=FILTER, line=dict(width=3))),
    ("output value it produces", dict(symbol="square", size=16, color=FILTER, opacity=0.85)),
    ("output", dict(symbol="square", size=16, color=OUTPUT)),
]
for name, marker in legend:
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=name, marker=marker))

fig.update_layout(
    width=900, height=980, plot_bgcolor="white", paper_bgcolor="white",
    font=dict(size=13), margin=dict(t=40, b=90, l=10, r=10),
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.02, yanchor="top"),
)
fig.show()

The weight shape `(32, 4, 3, 3)` shows that each of the 32 filters has a 3×3 kernel for **each**
of the 4 input channels. In 3D the kernel is 3×3×3, so one first-layer filter has 4·27 = 108 weights
(plus a bias).

## 3. Every convolution in the network

Forward hooks record each `Conv3d` (encoder, residual branches) and `ConvTranspose3d` (decoder
upsampling) during one forward pass. Layers with `residual` in the name are the shortcut branches
of residual units.

In [ ]:
rows = []


def make_hook(name):
    def hook(module, inputs, output):
        rows.append({
            "layer": name,
            "type": type(module).__name__,
            "in ch": module.in_channels,
            "out ch": module.out_channels,
            "kernel": module.kernel_size[0],
            "stride": module.stride[0],
            "output size": "×".join(map(str, output.shape[2:])),
            "params": sum(p.numel() for p in module.parameters()),
            "act MB": output.numel() * output.element_size() / 1e6,
        })
    return hook


handles = [m.register_forward_hook(make_hook(n)) for n, m in model.named_modules()
           if isinstance(m, (nn.Conv3d, nn.ConvTranspose3d))]

x = torch.randn(1, IN_CHANNELS, PATCH, PATCH, PATCH, device=DEVICE)

with torch.no_grad():
    out = model(x)
for h in handles:
    h.remove()

layers = pd.DataFrame(rows)
layers.index.name = "order"
print(f"{len(layers)} convolution layers, input {tuple(x.shape)} -> output {tuple(out.shape)}")
pd.set_option("display.max_rows", 100)

layers

23 convolution layers, input (1, 4, 64, 64, 64) -> output (1, 4, 64, 64, 64)


,layer,type,in ch,out ch,kernel,stride,output size,params,act MB
order,,,,,,,,,
0,model.0.residual,Conv3d,4,32,3,2,32×32×32,3488,4.194304
1,model.0.conv.unit0.conv,Conv3d,4,32,3,2,32×32×32,3488,4.194304
2,model.0.conv.unit1.conv,Conv3d,32,32,3,1,32×32×32,27680,4.194304
3,model.1.submodule.0.residual,Conv3d,32,64,3,2,16×16×16,55360,1.048576
4,model.1.submodule.0.conv.unit0.conv,Conv3d,32,64,3,2,16×16×16,55360,1.048576
5,model.1.submodule.0.conv.unit1.conv,Conv3d,64,64,3,1,16×16×16,110656,1.048576
6,model.1.submodule.1.submodule.0.residual,Conv3d,64,128,3,2,8×8×8,221312,0.262144
7,model.1.submodule.1.submodule.0.conv.unit0.conv,Conv3d,64,128,3,2,8×8×8,221312,0.262144
8,model.1.submodule.1.submodule.0.conv.unit1.conv,Conv3d,128,128,3,1,8×8×8,442496,0.262144


### Parameters per resolution level

Most parameters sit in the deep, low-resolution levels (many channels), while most
activation memory sits in the high-resolution levels (large maps).

In [6]:
per_level = (layers.groupby("output size", sort=False)
             .agg(layers=("layer", "count"), params=("params", "sum"), act_MB=("act MB", "sum")))
per_level["params %"] = 100 * per_level["params"] / per_level["params"].sum()
per_level.round(1)

,layers,params,act_MB,params %
output size,,,,
32×32×32,5,172960,21.0,1.3
16×16×16,5,774464,5.2,6.0
8×8×8,5,3318400,1.3,25.8
4×4×4,6,8599232,0.4,66.8
64×64×64,2,7352,8.4,0.1


## 4. Memory

- **Parameters**: weights × 4 bytes (float32). Gradients and Adam's two moment buffers add
  roughly 3× more during training.
- **Activations**: the sum of conv outputs above is only a rough lower bound. Normalisation,
  activation layers and skip concatenations also keep tensors for the backward pass. For the
  real number, run `python -m src.model` (or `scripts/check_model.py`) on the GPU.

In [ ]:
n_params = count_parameters(model)
print(f"parameters:              {n_params / 1e6:.2f} M = {n_params * 4 / 1e6:.0f} MB (fp32)")
print(f"+ gradients + Adam state: ~{n_params * 4 * 4 / 1e6:.0f} MB in total")

act = layers["act MB"].sum()
print(f"conv outputs, patch {PATCH}: {act:.0f} MB per sample (rough lower bound)")
if PATCH != 128:
    print(f"  scaled to patch 128:    ~{act * (128 / PATCH) ** 3:.0f} MB per sample")

parameters:              12.87 M = 51 MB (fp32)
+ gradients + Adam state: ~206 MB in total
conv outputs, patch 64: 36 MB per sample (rough lower bound)
  scaled to patch 128:    ~291 MB per sample


## 5. Output: logits, sigmoid, not softmax

The network returns **logits** (raw values, no sigmoid applied inside the model), one channel per region. Regions overlap:
a voxel of enhancing tissue (label 3) belongs to ET, TC and WT at the same time.

- **Sigmoid** treats each channel separately: each gives its own probability in [0, 1],
  so several channels can be close to 1 for the same voxel.
- **Softmax** would force the channel probabilities of a voxel to sum to 1, i.e. exactly one
  region per voxel, which contradicts ET $\subset$ TC $\subset$ WT.

Final mask: `sigmoid(logits) > 0.5`, per channel.

In [ ]:
regions = ["ET", "TC", "WT", "RC"][: model.out_channels]
logits = torch.tensor([4.0, 3.5, 5.0, -6.0][: model.out_channels])  # one ET voxel, as a trained model might output

table = pd.DataFrame({
    "logit": logits.numpy(),
    "sigmoid": torch.sigmoid(logits).numpy(),
    "softmax": torch.softmax(logits, dim=0).numpy(),
}, index=regions)

table["mask (sigmoid > 0.5)"] = table["sigmoid"] > 0.5
table.round(3)

,logit,sigmoid,softmax,mask (sigmoid > 0.5)
ET,4.0,0.982,0.231,True
TC,3.5,0.971,0.140,True
WT,5.0,0.993,0.629,True
RC,-6.0,0.002,0.000,False


With sigmoid the voxel is correctly in ET, TC and WT. With softmax the probabilities share one
budget of 1, so at most one region could pass a 0.5 threshold.